# Pandas

# First, I import pandas and read my data "citibikedata.csv". Then I use several codes to read my data.

In [9]:
import pandas as pd

df = pd.read_csv("citibikedata.csv")
print("shape of data:", df.shape)
print("first five row: ", df.head())
print(df.info())
df.columns.tolist()

shape of data: (116071, 13)
first five row:              ride_id  rideable_type               started_at  \
0  3AAD1E1BCB6B326B  electric_bike  2025-09-04 08:36:52.378   
1  EC348BE3649505F6  electric_bike  2025-09-08 18:00:44.089   
2  6639ACBA00F4DC07   classic_bike  2025-09-06 08:59:06.165   
3  A10A50B94E43F61E   classic_bike  2025-09-08 18:11:33.185   
4  56D1E808E38A2034  electric_bike  2025-09-24 17:39:37.945   

                  ended_at     start_station_name start_station_id  \
0  2025-09-04 08:47:21.064           Lincoln Park            JC053   
1  2025-09-08 18:18:48.698            Exchange Pl            JC116   
2  2025-09-06 14:27:30.692  Union St & Bergen Ave            JC122   
3  2025-09-08 18:17:06.671   Bergen Ave & Sip Ave            JC109   
4  2025-09-24 18:01:08.535            Exchange Pl            JC116   

  end_station_name end_station_id  start_lat  start_lng    end_lat    end_lng  \
0           Dey St          JC065  40.724605 -74.078406  40.737711 -74.066

['ride_id',
 'rideable_type',
 'started_at',
 'ended_at',
 'start_station_name',
 'start_station_id',
 'end_station_name',
 'end_station_id',
 'start_lat',
 'start_lng',
 'end_lat',
 'end_lng',
 'member_casual']

# I choose to analysis tripduration time, so I need to calculate it by using "started_at" and "ended_at".

In [15]:
df["started_at"] = pd.to_datetime(df["started_at"])
df["ended_at"] = pd.to_datetime(df["ended_at"])
df["tripduration"] = (df["ended_at"] - df["started_at"]).dt.total_seconds()
trip_durations = df["tripduration"]

# After I get new variable trip_duration, I begin calculate average, median and mode by using pandas.

In [17]:
print(f"average: {trip_durations.mean():.2f} s")
print(f"median: {trip_durations.median():.2f} s")
modes = trip_durations.mode()
print(f"mode: {modes.values[0]:.2f} s")

average: 647.80 s
median: 388.37 s
mode: 284.73 s


# Python

# The first step is to read data

In [22]:
import csv
from datetime import datetime

durations = []
with open("citibikedata.csv", "r") as file:
    reader = csv.DictReader(file)
    for row in reader:
        try:
            start_str = row["started_at"].replace("Z", "+00:00")
            end_str = row["ended_at"].replace("Z", "+00:00")
            start_time = datetime.fromisoformat(start_str)
            end_time = datetime.fromisoformat(end_str)
            duration = (end_time - start_time).total_seconds()
            if 60 < duration < 24 * 3600:
                durations.append(duration)
        except:
            continue
print("valid data:", {len(durations)})

valid data: {116033}


# In order to calculate the mean, median and mode, I directly define three functions to do so.

In [26]:
def c_mean(data):
    return sum(data) / len(data)


def c_median(data):
    sorted_data = sorted(data)
    n = len(sorted_data)
    mid = n // 2
    if n % 2 == 0:
        return (sorted_data[mid - 1] + sorted_data[mid]) / 2
    else:
        return sorted_data[mid]


def c_mode(data, bin_size=60):
    bins = {}
    for value in data:
        bin_key = round(value / bin_size) * bin_size
        bins[bin_key] = bins.get(bin_key, 0) + 1

    max_bin = max(bins, key=bins.get)
    return max_bin, bins[max_bin]


# Now I use these functions to calculate.

In [27]:
mean_val = c_mean(durations)
median_val = c_median(durations)
mode_bin, mode_count = c_mode(durations)


# Finally, I print three numbers.

In [28]:
print(f"average: {mean_val / 60:.2f} min")
print(f"median: {median_val / 60:.2f} min")
print(f"mode: {mode_bin / 60:.1f} min")

average: 10.31 min
median: 6.47 min
mode: 4.0 min


# Visualization

# First I define Time Bins. Second, I count Durations in each bin and find maximum count. Third, I generate visual output by using visual representations, alignment and so on.

In [33]:
def create_visualization(durations):
    bins = {
        (0, 300): 0,
        (300, 600): 0,
        (600, 900): 0,
        (900, 1800): 0,
        (1800, 3600): 0,
        (3600, 7200): 0,
        (7200, 18000): 0,
        (18000, float("inf")): 0,
    }

    for duration in durations:
        for bin_range in bins:
            if bin_range[0] <= duration < bin_range[1]:
                bins[bin_range] += 1
                break

    max_count = max(bins.values())

    if max_count == 0:
        print("Not Found")
        return

    for (start, end), count in sorted(bins.items()):
        if count > 0:
            bar_length = int((count / max_count) * 40)
            bar = "■" * bar_length

            start_min = start / 60
            end_min = end / 60
            if end == float("inf"):
                label = f"{start_min:.0f}min+"
            else:
                label = f"{start_min:.0f}-{end_min:.0f}min"

            percentage = (count / len(durations)) * 100
            print(f"{label:15} | {bar} {count} ({percentage:.1f}%)")


create_visualization(durations)

0-5min          | ■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■ 40062 (34.5%)
5-10min         | ■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■ 45171 (38.9%)
10-15min        | ■■■■■■■■■■■■■■ 15865 (13.7%)
15-30min        | ■■■■■■■■■ 10483 (9.0%)
30-60min        | ■■ 3235 (2.8%)
60-120min       |  918 (0.8%)
120-300min      |  183 (0.2%)
300min+         |  116 (0.1%)
